In [0]:
%run
../00-common/01.environment-config

In [0]:
from pyspark.sql.functions import col, initcap

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"
# Read results data from bronze layer
bronze_results_df = spark.table(bronze_table)
# display(bronze_results_df)

In [0]:
results_df = (
    bronze_results_df
    .drop("url")
    .withColumnRenamed("constructorId", "constructor_id")
    .withColumnRenamed("driverId", "driver_id")
    .withColumnRenamed("raceName", "race_name")
    .withColumnRenamed("positionText", "final_position_text")
)

In [0]:
results_df = (
    results_df
    .withColumnRenamed("date", "race_date")
    .withColumnRenamed("grid", "grid_position")
    .withColumnRenamed("laps", "completed_laps")
    .withColumnRenamed("number", "car_number")
    .withColumnRenamed("position", "final_position")
)

In [0]:
results_df = (
    results_df
    .filter(
        col("season").isNotNull() &
        col("round").isNotNull() &
        col("constructor_id").isNotNull() &
        col("driver_id").isNotNull()
    )
)

In [0]:
results_df = results_df.dropDuplicates(["season", "round", "constructor_id", "driver_id"])

In [0]:
results_df = results_df.withColumn("race_name", initcap(col("race_name")))

In [0]:
results_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

In [0]:
%sql
-- select * from formula1.silver.results limit 10;